* check RAÄ status

In [1]:
import time
from urllib.error import HTTPError
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone


def get_results(endpoint_url, query):
    user_agent = (
        "Salgo60-LinkCheck/1.0 "
        "(https://www.wikidata.org/wiki/User:Salgo60)"
    )

    sparql = SPARQLWrapper(endpoint_url, agent=user_agent)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.setTimeout(90)

    for attempt in range(4):
        try:
            return sparql.query().convert()

        except HTTPError as error:
            # Övriga HTTP-fel, eller sista försöket: visa felet.
            if error.code != 429 or attempt == 3:
                raise

            retry_after = error.headers.get("Retry-After")
            wait_seconds = 60

            if retry_after:
                try:
                    wait_seconds = float(retry_after)
                except ValueError:
                    retry_at = parsedate_to_datetime(retry_after)
                    wait_seconds = (
                        retry_at - datetime.now(timezone.utc)
                    ).total_seconds()

            # Vänta minst en minut, med lite säkerhetsmarginal.
            wait_seconds = max(60, wait_seconds) + 2

            print(
                f"Wikidata svarade 429. Väntar {wait_seconds:.0f} "
                f"sekunder före nytt försök ({attempt + 2}/4).",
                flush=True,
            )
            time.sleep(wait_seconds)

In [2]:
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from urllib.error import HTTPError
import json
import time

import urllib3
from SPARQLWrapper import SPARQLWrapper, JSON
from tqdm.auto import tqdm


endpoint_url = "https://query.wikidata.org/sparql"

USER_AGENT = (
    "Salgo60-LinkCheck/1.0 "
    "(https://www.wikidata.org/wiki/User:Salgo60)"
)

http = urllib3.PoolManager(
    headers={"User-Agent": USER_AGENT},
    timeout=urllib3.Timeout(connect=10, read=30),
)

query = """
SELECT ?item ?itemLabel ?Ksam ?oldURL WHERE {
  ?item wdt:P1260 ?Ksam .
  BIND(URI(CONCAT("http://kulturarvsdata.se/", ?Ksam)) AS ?oldURL)
  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "sv,en".
  }
}
"""
query = """
SELECT DISTINCT ?item ?Ksam ?oldURL WHERE {
  ?item wdt:P1260 ?Ksam .
  BIND(
    URI(CONCAT("http://kulturarvsdata.se/", ?Ksam))
    AS ?oldURL
  )
}
"""


def get_results(endpoint_url, query):
    batch_size = 1000
    offset = 0
    all_rows = []

    with tqdm(desc="Hämtar Wikidata", unit="poster") as progress:
        while True:
            page_query = (
                query
                + "\nORDER BY STR(?item) STR(?Ksam)"
                + f"\nLIMIT {batch_size}"
                + f"\nOFFSET {offset}"
            )

            for attempt in range(4):
                sparql = SPARQLWrapper(
                    endpoint_url,
                    agent=USER_AGENT,
                )
                sparql.setQuery(page_query)
                sparql.setReturnFormat(JSON)
                sparql.setTimeout(90)

                try:
                    response = sparql.query().response
                    try:
                        raw = response.read().decode("utf-8")
                    finally:
                        response.close()

                    # Acceptera kontrolltecken i strängar.
                    # Andra syntaxfel leder till nytt försök.
                    data = json.loads(raw, strict=False)

                    if not isinstance(data, dict):
                        raise ValueError("Svaret är inte ett JSON-objekt.")

                    rows = data.get("results", {}).get("bindings")

                    if not isinstance(rows, list):
                        raise ValueError("Svaret saknar resultatlistan.")

                    if any(
                        not isinstance(row, dict)
                        or not all(
                            key in row
                            and isinstance(row[key], dict)
                            and "value" in row[key]
                            for key in ("item", "Ksam", "oldURL")
                        )
                        for row in rows
                    ):
                        raise ValueError("Svaret innehåller ofullständiga poster.")

                    break

                except HTTPError as error:
                    if (
                        error.code not in (429, 502, 503, 504)
                        or attempt == 3
                    ):
                        raise

                    delay = retry_delay(error.headers)
                    tqdm.write(
                        f"HTTP {error.code}, offset {offset}. "
                        f"Väntar {delay:.0f} sekunder."
                    )
                    time.sleep(delay)

                except (ValueError, UnicodeDecodeError) as error:
                    if attempt == 3:
                        raise RuntimeError(
                            f"Kunde inte läsa portionen vid offset {offset}. "
                            "Hämtningen avbryts så att inga poster "
                            "hoppas över."
                        ) from error

                    tqdm.write(
                        f"Ogiltigt svar vid offset {offset}: {error}\n"
                        "Försöker samma portion igen om 62 sekunder."
                    )
                    time.sleep(62)

            all_rows.extend(rows)
            progress.update(len(rows))
            progress.set_postfix(portion=offset // batch_size + 1)

            if len(rows) < batch_size:
                break

            offset += batch_size

            # Håll högst ett WDQS-anrop per minut.
            progress.set_description("Väntar inför nästa hämtning")
            time.sleep(62)
            progress.set_description("Hämtar Wikidata")

    return {"results": {"bindings": all_rows}}  
    
def retry_delay(headers):
    """Läs Retry-After som sekunder eller HTTP-datum."""
    value = headers.get("Retry-After")

    if value:
        try:
            return max(60, float(value)) + 2
        except ValueError:
            try:
                retry_at = parsedate_to_datetime(value)
                seconds = (
                    retry_at - datetime.now(timezone.utc)
                ).total_seconds()
                return max(60, seconds) + 2
            except (ValueError, TypeError, OverflowError):
                pass

    return 62


def get_results(endpoint_url, query):
    sparql = SPARQLWrapper(endpoint_url, agent=USER_AGENT)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    sparql.setTimeout(90)

    for attempt in range(4):
        try:
            response = sparql.query().response
            try:
                raw = response.read().decode("utf-8")
            finally:
                response.close()

            try:
                data = json.loads(raw)
            except json.JSONDecodeError as error:
                if not error.msg.startswith("Invalid control character"):
                    raise

                print(
                    "JSON-svaret innehåller ett oescapat kontrolltecken "
                    f"på rad {error.lineno}. Läser med strict=False.",
                    flush=True,
                )
                data = json.loads(raw, strict=False)

            if not isinstance(data, dict) or "bindings" not in data.get("results", {}):
                raise ValueError("Svaret saknar förväntade SPARQL-resultat.")

            return data

        except HTTPError as error:
            if error.code not in (429, 502, 503, 504) or attempt == 3:
                raise

            delay = retry_delay(error.headers)
            print(
                f"Wikidata: HTTP {error.code}. "
                f"Väntar {delay:.0f} sekunder före nytt försök.",
                flush=True,
            )
            time.sleep(delay)


def check(url):
    """Returnera slutlig HTTP-status efter omdirigeringar."""
    try:
        response = http.request(
            "GET",
            url,
            redirect=True,
            retries=urllib3.Retry(
                total=2,
                redirect=5,
                backoff_factor=1,
                status=0,
                respect_retry_after_header=True,
                raise_on_redirect=False,
                raise_on_status=False,
            ),
        )
        try:
            return response.status, None
        finally:
            response.release_conn()

    except urllib3.exceptions.HTTPError as error:
        return None, str(error)


start_time = datetime.now()
print("Start:", start_time)
print("Hämtar från Wikidata …", flush=True)

results = get_results(endpoint_url, query)
rows = results["results"]["bindings"]

print(f"Hämtade {len(rows):,} länkar.")

ok = 0
notok = 0
uncertain = 0

# Sparas i minnet för fortsatt analys i notebooken.
problems = []

with tqdm(rows, desc="Kontrollerar länkar", unit="länk") as progress:
    for row in progress:
        url = row["oldURL"]["value"]
        status, error = check(url)

        if status == 200:
            ok += 1
        else:
            # Nätverksfel och serverbegränsningar bevisar inte länkröta.
            is_uncertain = (
                status is None
                or status in (403, 408, 425, 429)
                or status >= 500
            )

            if is_uncertain:
                uncertain += 1
            else:
                notok += 1

            problems.append({
                "item": row["item"]["value"],
                "label": row.get("itemLabel", {}).get("value", ""),
                "url": url,
                "status": status,
                "error": error,
            })

            # tqdm.write(f"{status or 'Nätverksfel'}: {url}")

        progress.set_postfix(
            OK=ok,
            ej_200=notok,
            osäkra=uncertain,
        )

        # Respektera eventuell begränsning hos länkens server.
        if status == 429:
            tqdm.write("HTTP 429: pausar länkkontrollen i 62 sekunder.")
            time.sleep(62)
        else:
            time.sleep(0.2)

print()
print(f"OK: {ok}")
print(f"Övriga HTTP-svar: {notok}")
print(f"Osäkra / behöver kontrolleras igen: {uncertain}")
print("Slut:", datetime.now())
print("Tidsåtgång:", datetime.now() - start_time)

Start: 2026-10-16 06:03:23.184951
Hämtar från Wikidata …
Hämtade 318,294 länkar.


Kontrollerar länkar:   0%|          | 0/318294 [00:00<?, ?länk/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter serve

In [3]:
print(f"OK: {ok}")
print(f"Övriga HTTP-svar: {notok}")
print(f"Osäkra / behöver kontrolleras igen: {uncertain}")
print("Slut:", datetime.now())
print("Tidsåtgång:", datetime.now() - start_time)

OK: 162334
Övriga HTTP-svar: 155957
Osäkra / behöver kontrolleras igen: 3
Slut: 2026-10-17 09:48:25.280504
Tidsåtgång: 1 day, 3:45:02.096319
